# Week 12 -- Model Monitoring, Drift & Performance Degradation

**FraudFlow Project | Week 12**

Monitoring the frozen Week 8 XGBoost model in production (historical demonstration mode). Detecting shifts in feature distributions, prediction anomalies, and measuring performance/cost degradation.


## 1. Objective and Monitoring Architecture

- **Data Quality:** Prevent broken inference (missing features, unexpected nulls).
- **Feature Drift:** Detect structural changes in incoming transaction behavior (PSI, KS Test).
- **Prediction Drift:** Monitor shift in model output probabilities.
- **Performance & Cost:** Continuously evaluate PR-AUC and Operational Cost using delayed ground-truth labels.
- **Alerts:** Generate severity-based alerts (INFO, WARNING, CRITICAL).


## 2. Load Reference & Current Windows

For this demonstration, we sample from the canonical training set (Reference) and the untouched test set (Current). No retraining occurs, and the threshold is frozen at 0.11.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

reports_dir = Path('reports/monitoring')
print('Monitoring architecture uses historical data replay.')
with open(reports_dir / 'latest_monitoring_report.json') as f:
    report = json.load(f)

print(f"Run ID: {report['run_id']}")
print(f"Reference Window Size: {report['reference_window_size']}")
print(f"Current Window Size: {report['current_window_size']}")

## 3. Alerts Summary

Reviewing any Data Quality, Feature Drift, or Performance degradation warnings.


In [ ]:
alerts = pd.DataFrame(report['alerts'])
if alerts.empty:
    print('No alerts triggered! System is healthy.')
else:
    display(alerts[['severity', 'category', 'rule', 'observed_value', 'configured_threshold']])

## 4. Feature Drift

Evaluating Population Stability Index (PSI) and Kolmogorov-Smirnov (KS) test values for numerical features.


In [ ]:
drift_df = pd.read_csv(reports_dir / 'feature_drift_report.csv')
drift_df = drift_df.sort_values('psi', ascending=False)
display(drift_df.head(10))
print('Features with highest PSI indicate the largest distributional shift.')

## 5. Prediction Distribution Monitoring


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(reports_dir / 'figures/prediction_drift.png')))
print(f"Reference Mean Prob: {report['prediction_drift']['ref_mean_prob']:.4f}")
print(f"Current Mean Prob:   {report['prediction_drift']['cur_mean_prob']:.4f}")
print(f"KS p-value:          {report['prediction_drift']['ks_pvalue']:.4e}")

## 6. Labeled Performance & Cost Monitoring

When labels arrive, evaluate the model's actual effectiveness and business cost.


In [ ]:
perf = report['performance']
cost = report['cost']

print('--- Performance Metrics ---')
print(f"PR-AUC:    {perf['pr_auc']:.4f}")
print(f"Precision: {perf['precision']:.4f}")
print(f"Recall:    {perf['recall']:.4f}")
print(f"F1 Score:  {perf['f1']:.4f}")

print('\n--- Monetary Cost ---')
print(f"False Positives: {cost['fp_count']} (Cost: ${cost['cost_fp']:,.2f})")
print(f"False Negatives: {cost['fn_count']} (Cost: ${cost['cost_fn']:,.2f})")
print(f"Total Cost:      ${cost['total_cost']:,.2f}")
print(f"Cost per Tx:     ${cost['cost_per_tx']:.4f}")

## 7. Limitations & Operational Recommendations

1. **False Positives in Drift:** A statistically significant KS test does not inherently mean model decay; verify with PR-AUC when labels are available.
2. **Delayed Labels:** Performance metrics strictly depend on mature, fully-settled fraud labels. Unlabeled windows can only leverage PSI and Prediction Drift as early-warning indicators.
3. **No Automated Actions:** Alerts are designed for human-in-the-loop ML operations. Do not blindly auto-retrain a model on a drift alert.
